# bin() and Number Base Prefixes
Convert integers to binary strings, inspect the prefix, and try binary addition.

**How to use:** In Google Colab, choose **Runtime → Run all**, then edit the input fields. You can also change any code cell and run it again. No account credentials or external data are required by this notebook.

## 1. A value and its representation
An integer is a numeric value. Binary, decimal, octal, and hexadecimal are ways to represent that value. `int("1010", 2)` returns the integer 10; it does not return a decimal string.

In [ ]:
print(int("1010", 2))        # 10
print(bin(10))               # '0b1010'
print(type(bin(10)))         # <class 'str'>
print(bin(10)[2:])           # '1010'
print(type(bin(10)[2:]))     # <class 'str'>

## 2. What is a prefix?
A prefix is a marker at the beginning of a representation. Here it identifies the base.

| Prefix | Base | Integer literal | Decimal value |
|---|---|---|---|
| `0b` | Binary, base 2 | `0b1010` | 10 |
| `0o` | Octal, base 8 | `0o12` | 10 |
| `0x` | Hexadecimal, base 16 | `0xA` | 10 |

Unquoted `0b1010` is an integer literal. Quoted `"0b1010"` is a string. The prefix identifies the base; it is not one of the digits contributing to the value.

In [ ]:
print(0b1010 == 0o12 == 0xA == 10)  # True
print(bin(10), oct(10), hex(10))
print(type(0b1010), type("0b1010"))

## 3. Interactive representation explorer
Change the integer. The display updates automatically. Try 0, 5, 10, 21, and -10. The bit diagram shows the magnitude; the sign is displayed separately.

In [ ]:
import html
import ipywidgets as widgets
from IPython.display import HTML, display

# Colab uses this manager to render interactive widgets.
try:
    from google.colab import output as colab_output
    colab_output.enable_custom_widget_manager()
except ImportError:
    pass

number_input = widgets.IntText(value=10, description="Integer:")
number_output = widgets.Output()

def render_number(change=None):
    value = number_input.value
    binary = bin(value)
    digits = binary.removeprefix("-").removeprefix("0b")
    sign = "-" if value < 0 else ""
    bits = ""
    if len(digits) <= 32:
        for index, digit in enumerate(digits):
            power = len(digits) - index - 1
            bits += (
                '<div style="padding:10px;text-align:center;border:1px solid #cbd5e1;'
                'border-radius:8px"><b>' + digit + '</b><br>'
                '<small>2<sup>' + str(power) + '</sup> = ' + str(2**power) + '</small></div>'
            )
    else:
        bits = "<p>Place-value cards are shown for up to 32 bits.</p>"
    body = (
        '<div style="font-family:system-ui;padding:16px">'
        '<h3>One integer, several representations</h3>'
        '<table><tr><th style="padding:8px">Function</th><th>Returned string</th></tr>'
    )
    for label, text in [("bin()", binary), ("oct()", oct(value)), ("hex()", hex(value))]:
        body += '<tr><td style="padding:8px">' + label + '</td><td><code>' + html.escape(repr(text)) + '</code></td></tr>'
    body += (
        '</table><p>Binary: <code>' + sign +
        '<span style="background:#dbeafe;padding:4px">0b</span>' + digits +
        '</code> — highlighted prefix; result type: <b>str</b></p>'
        '<div style="display:flex;gap:6px;flex-wrap:wrap">' + bits + '</div></div>'
    )
    with number_output:
        number_output.clear_output(wait=True)
        display(HTML(body))

number_input.observe(render_number, names="value")
display(number_input, number_output)
render_number()

## 4. Why use [2:]?
String indices begin at 0. In `"0b1010"`, indices 0 and 1 contain the prefix. `[2:]` returns the substring from index 2 to the end.

This works for the **nonnegative** result in LeetCode 67. For a negative value, `bin(-10)` returns `"-0b1010"`, so `[2:]` produces `"b1010"`. `format(value, "b")` gives a binary string without the base prefix and preserves a minus sign.

In [ ]:
binary = bin(10)
print(list(enumerate(binary)))
print(binary[2:])                # '1010'
print(bin(0)[2:])                # '0'
print(bin(-10), bin(-10)[2:])    # '-0b1010', 'b1010'
print(format(-10, "b"))          # '-1010'

## 5. Interactive Add Binary lab
Enter two nonempty strings containing only 0 and 1. Watch each conversion, the integer sum, and the returned binary string. Try `11 + 1`, `1010 + 1011`, and `0 + 0`.

The lab accepts up to 10,000 bits per input. It accepts leading zeroes for exploration; the result has no unnecessary leading zeroes.

In [ ]:
a_input = widgets.Text(value="1010", description="Binary a:")
b_input = widgets.Text(value="1011", description="Binary b:")
addition_output = widgets.Output()

def render_addition(change=None):
    a, b = a_input.value, b_input.value
    with addition_output:
        addition_output.clear_output(wait=True)
        if not a or not b or any(c not in "01" for c in a + b):
            print("Enter a nonempty string containing only 0 and 1 in each field.")
            return
        if max(len(a), len(b)) > 10000:
            print("Use at most 10,000 bits per input.")
            return
        num_a = int(a, 2)
        num_b = int(b, 2)
        total = num_a + num_b
        prefixed = bin(total)
        result = prefixed[2:]
        print("1. int(a, 2) =", num_a, "| type:", type(num_a).__name__)
        print("2. int(b, 2) =", num_b, "| type:", type(num_b).__name__)
        print("3. Integer sum =", total)
        print("4. bin(sum) =", repr(prefixed))
        print("5. bin(sum)[2:] =", repr(result), "| type:", type(result).__name__)

a_input.observe(render_addition, names="value")
b_input.observe(render_addition, names="value")
display(widgets.VBox([a_input, b_input]), addition_output)
render_addition()

## 6. The LeetCode solution
This is the expanded conversion approach. Python handles the integer addition, including carry propagation. Binary conversion and addition still process the input bits; the solution takes O(m + n) time and space for input lengths m and n.

In [ ]:
class Solution:
    def addBinary(self, a: str, b: str) -> str:
        num_a = int(a, 2)
        num_b = int(b, 2)
        add_ab = num_a + num_b
        return bin(add_ab)[2:]

solution = Solution()
for a, b in [("11", "1"), ("1010", "1011"), ("0", "0")]:
    print(a, "+", b, "=", solution.addBinary(a, b))

## 7. Try it yourself
Predict each answer before running the cell. Edit the inputs to make your own examples.

- What type does `bin()` return?
- Which characters does `[2:]` remove for a nonnegative integer?
- Do `0b1010` and `10` represent different values?
- Why is the result of `1111 + 1` one bit longer?

In [ ]:
assert isinstance(bin(10), str)
assert int("1010", 2) == 10
assert 0b1010 == 0o12 == 0xA == 10
assert solution.addBinary("1111", "1") == "10000"
assert solution.addBinary("0", "0") == "0"
print("All checks passed. Change the examples and run again!")

## Remember
- `int(text, 2)`: binary string → integer.
- `bin(value)`: integer → binary string with a `0b` prefix.
- `[2:]`: remove the first two characters, suitable here for nonnegative values.
- `0b`, `0o`, and `0x` identify binary, octal, and hexadecimal representations.
- Python integers support arbitrary precision; binary conversion is not subject to the decimal string conversion digit limit.

Sources: [Python bin()](https://docs.python.org/3/library/functions.html#bin), [int()](https://docs.python.org/3/library/functions.html#int), [Integer literals](https://docs.python.org/3/reference/lexical_analysis.html#integer-literals), [String slicing](https://docs.python.org/3/library/stdtypes.html#common-sequence-operations).

Practice: [LeetCode 67 — Add Binary notes and solution](https://github.com/Lawson-Dong/leetcode-solution/tree/main/easy/067).